# Linear Regression
## Real-world scenario: Predicting house prices

A real-estate agency wants to estimate the **selling price** of a house from a few features such as its size, number of bedrooms and age. Price is a continuous number, so this is a **regression** problem and Linear Regression is a natural first model.

We will go step by step: create data -> clean it -> split it -> train -> evaluate.

### Step 1 - Import the libraries we need

In [ ]:
# pandas / numpy handle the data, matplotlib draws charts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# scikit-learn gives us the model, the train/test split and the metrics
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)  # makes the random numbers reproducible for everyone

### Step 2 - Create a small, realistic dataset
We build 40 houses. Notice we deliberately add a couple of **missing values** and a **duplicate row** so we have something real to clean in the next step.

In [ ]:
n = 40
# Realistic-looking features
size_sqft   = np.random.randint(600, 3000, n)          # living area
bedrooms    = np.random.randint(1, 6, n)               # number of bedrooms
age_years   = np.random.randint(0, 50, n)              # age of the house

# Price is roughly driven by these features + some random noise (real market variation)
price = (50 * size_sqft) + (15000 * bedrooms) - (800 * age_years) \
        + np.random.normal(0, 20000, n) + 30000

df = pd.DataFrame({
    'size_sqft': size_sqft,
    'bedrooms': bedrooms,
    'age_years': age_years,
    'price': price.round(0)
})

# Inject a few messy values on purpose so we can practise cleaning
df.loc[3, 'size_sqft'] = np.nan       # missing size
df.loc[7, 'age_years'] = np.nan       # missing age
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)  # a duplicate row

df.head()

### Step 3 - Explore the data
Always look before you model: how big is it, what types, where are the gaps?

In [ ]:
print('Shape (rows, columns):', df.shape)
print('\nMissing values per column:')
print(df.isnull().sum())
print('\nNumber of duplicate rows:', df.duplicated().sum())
df.describe()

### Step 4 - Clean the data
We remove duplicates and fill the missing numbers with the column **median** (median is safer than mean when there are outliers).

In [ ]:
# 1) drop exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# 2) fill missing numeric values with the column median
df['size_sqft'] = df['size_sqft'].fillna(df['size_sqft'].median())
df['age_years'] = df['age_years'].fillna(df['age_years'].median())

# confirm the data is now clean
print('Missing values remaining:', df.isnull().sum().sum())
print('Duplicates remaining:', df.duplicated().sum())

### Step 5 - Split into features (X) and target (y)
`X` = the inputs the model learns from, `y` = the value we want to predict.

In [ ]:
X = df[['size_sqft', 'bedrooms', 'age_years']]   # inputs
y = df['price']                                  # what we predict
print('X shape:', X.shape, '| y shape:', y.shape)

### Step 6 - Train / test split
We hold back 20% of the houses as a **test set** the model never sees during training, so we can measure how well it generalises.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train), '| Testing rows:', len(X_test))

### Step 7 - Train the model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)   # the model learns the best-fit line here

# The learned relationship: price = intercept + coef*feature ...
for feature, coef in zip(X.columns, model.coef_):
    print(f'{feature:12s}: {coef:,.2f}')
print('Intercept   :', round(model.intercept_, 2))

### Step 8 - Evaluate on the test set
For regression we look at the average error (MAE / RMSE, in dollars) and R2 (how much of the price variation the model explains, 1.0 = perfect).

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE  (avg error): ${mae:,.0f}')
print(f'RMSE            : ${rmse:,.0f}')
print(f'R2  score       : {r2:.3f}')

# Visual check: predicted vs actual (points near the diagonal = good)
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual price'); plt.ylabel('Predicted price')
plt.title('Predicted vs Actual house prices'); plt.show()

### Step 9 - Predict the price of a brand-new house
A 1800 sqft, 3-bedroom, 10-year-old house:

In [ ]:
new_house = pd.DataFrame({'size_sqft': [1800], 'bedrooms': [3], 'age_years': [10]})
predicted = model.predict(new_house)[0]
print(f'Estimated price: ${predicted:,.0f}')